# DBSCAN_noise — copia de trabajo del equipo

**Original:** `DBSCAN_noise.ipynb` del Dr. J. Octavio Gutiérrez García — <https://github.com/octavio-gutierrez/computoparalelo2026b/blob/main/DBSCAN_noise.ipynb> (commit `dff4ef2d03db72d19facfc2d535a1fcd2bd6f1f0`, blob `992a6b0eeafc89a5a2c35074127af38dc9c1736f`). La copia sin modificar está en `notebooks/DBSCAN_noise.ipynb`. Uso para el curso; el repositorio original no declara licencia.

**Cambios respecto al original** (todas las demás instrucciones del profesor son idénticas):
1. **W1** (nueva): rutas `DATA_DIR` y `RESULTS_DIR`.
2. La celda 0 del profesor se dividió en **A** (generación de datos) y **B** (DBSCAN de scikit-learn + gráfica) sin cambiar sus instrucciones; en A, `np.savetxt` escribe en `DATA_DIR`.
3. En la celda 1 del profesor (**C1**), `np.loadtxt` lee desde `RESULTS_DIR`.
4. Celdas del equipo marcadas `[EQUIPO]`: **T1** guarda la referencia binaria de scikit-learn, **T2** valida las 3 clases y **T3** grafica las 3 clases.

**Uso**
- Generar datos: W1 → A con `n_points` = 4000, 20000, 40000, 80000, 120000, 140000, 160000, 180000 y 200000 (un valor por ejecución).
- Referencia de scikit-learn: B → T1 solo para 4000, 20000 y 40000 (para N mayores es opcional y lento).
- Ver un resultado del programa C++: W1 → `n_points = N` → C1 (requiere `results/viz/<N>_results.csv`).
- Validar las 3 clases: T2 y T3 (requieren `results/viz/<N>_detailed.csv`).
- Antes de guardar el cuaderno, dejar `n_points = 4000`.

In [ ]:
# [EQUIPO] W1 — Rutas del proyecto (única celda agregada antes de las del profesor).
import os
import numpy as np                 # también se importa en A; aquí permite ejecutar C1/T3 sin regenerar datos
import matplotlib.pyplot as plt

DATA_DIR = "../data/"              # entradas <N>_data.csv generadas por la celda A
RESULTS_DIR = "../results/viz/"    # salidas del programa C++: <N>_results.csv y <N>_detailed.csv
os.makedirs(DATA_DIR + "reference", exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# [PROFESOR] celda 0, parte A (generación de datos) — única modificación: prefijo DATA_DIR en np.savetxt
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt
import numpy as np

# Indicate the number of points
n_points = 4000

# randomly generating data points and noise
points, y_true = make_blobs(n_samples=n_points,
                            centers=4,
                            cluster_std=0.06,
                            random_state=11,
                            center_box=(0, 1.0))

# only positive points and with three decimals
points = np.round(np.abs(points[:, ::-1]), 3)

# storing points into a csv file
np.savetxt(DATA_DIR + str(n_points)+"_data.csv", points, delimiter=",",  fmt="%.3f")

In [ ]:
# [PROFESOR] celda 0, parte B (DBSCAN de scikit-learn + gráfica) — sin modificaciones
# clustering and detecting noise with dbscan
clusters = DBSCAN(eps=0.03, min_samples=10).fit_predict(points)

# plotting noise
plt.figure()
plt.title("Detecting noise with DBSCAN")
plt.scatter(points[:, 0], points[:, 1], c=np.where(clusters==-1, 0, 1), s=50)
plt.xticks([])
plt.yticks([])
plt.box(False)
plt.show()

In [ ]:
# [EQUIPO] T1 — Guarda la referencia binaria de scikit-learn calculada en B (0 = ruido, 1 = no ruido).
np.savetxt(DATA_DIR + "reference/" + str(n_points) + "_sklearn_binary.csv",
           np.where(clusters==-1, 0, 1), fmt="%d")

In [ ]:
# [PROFESOR] celda 1 (gráfica del resultado C++) — única modificación: prefijo RESULTS_DIR en np.loadtxt
# loading results from c++ program into a numpy array
result = np.loadtxt(RESULTS_DIR + str(n_points)+"_results.csv", delimiter=",")

# plotting noise
plt.figure()
plt.title("Detecting noise with my DBSCAN")
plt.scatter(result.T[0], result.T[1], c=result.T[2], s=50)
plt.xticks([])
plt.yticks([])
plt.box(False)
plt.show()

In [ ]:
# [EQUIPO] T2 — Validación de 3 clases (requiere results/viz/<N>_detailed.csv del programa C++ en modo serial).
# Compara contra el gemelo numpy (misma aritmética IEEE) y contra scikit-learn; ninguna discrepancia se acepta automáticamente.
import sys
sys.path.insert(0, "../scripts")
from validate import oracle_report   # scripts/validate.py (milestone M1)
oracle_report(DATA_DIR + str(n_points) + "_data.csv", RESULTS_DIR + str(n_points) + "_detailed.csv")

In [ ]:
# [EQUIPO] T3 — Las 3 clases del programa C++: 0 = ruido, 1 = core de 1er grado, 2 = core de 2º grado (borde).
detailed = np.loadtxt(RESULTS_DIR + str(n_points) + "_detailed.csv", delimiter=",")
plt.figure()
plt.title("C++: ruido / core 1er grado / core 2º grado")
for clase, nombre, tam in ((1, "core 1er grado", 10), (2, "core 2º grado", 30), (0, "ruido", 30)):
    sel = detailed[:, 2] == clase
    plt.scatter(detailed[sel, 0], detailed[sel, 1], s=tam, label=nombre)
plt.legend()
plt.xticks([])
plt.yticks([])
plt.box(False)
plt.show()